In [ ]:
import cv2
import torch
import numpy as np
from utils.spectreTrack import SpectreTrack


class SpectreVideoTracker:
    def __init__(self, model_path, device="cuda:2"):
        self.device = device

        self.model = SpectreTrack().to(device)
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.eval()

        self.template = None
        self.state = None  # (cx, cy, w, h)

    def _crop(self, image, cx, cy, size):
        h, w = image.shape[:2]

        x1 = int(cx - size / 2)
        y1 = int(cy - size / 2)
        x2 = int(cx + size / 2)
        y2 = int(cy + size / 2)

        pad_l = max(0, -x1)
        pad_t = max(0, -y1)
        pad_r = max(0, x2 - w)
        pad_b = max(0, y2 - h)

        if pad_l or pad_t or pad_r or pad_b:
            image = cv2.copyMakeBorder(
                image,
                pad_t, pad_b, pad_l, pad_r,
                cv2.BORDER_CONSTANT,
                value=(127, 127, 127),
            )

            x1 += pad_l
            x2 += pad_l
            y1 += pad_t
            y2 += pad_t

        crop = image[y1:y2, x1:x2]
        return cv2.resize(crop, (256, 256)), (x1, y1)

    def init(self, frame, bbox):
        x, y, w, h = bbox
        cx = x + w / 2
        cy = y + h / 2

        self.state = (cx, cy, w, h)

        template_crop, _ = self._crop(frame, cx, cy, 128)
        self.template = self._to_tensor(template_crop)

    def _to_tensor(self, img):
        img = torch.tensor(img).permute(2, 0, 1).float() / 255.0
        img = img.unsqueeze(0).to(self.device)
        return img

    def track(self, frame):
        cx, cy, w, h = self.state

        search_crop, offset = self._crop(frame, cx, cy, 256)
        search_tensor = self._to_tensor(search_crop)

        with torch.no_grad():
            bbox, _, conf = self.model(
                self.template,
                search_tensor,
                history_buffer=None
            )

        bbox = bbox[0].cpu().numpy()  # normalized in search crop

        # convert to search crop pixel coords
        sx, sy = offset

        pred_cx = bbox[0] * 256 + sx
        pred_cy = bbox[1] * 256 + sy
        pred_w  = bbox[2] * 256
        pred_h  = bbox[3] * 256

        x = pred_cx - pred_w / 2
        y = pred_cy - pred_h / 2

        self.state = (pred_cx, pred_cy, pred_w, pred_h)

        return (int(x), int(y), int(pred_w), int(pred_h)), float(conf)

In [ ]:
cap = cv2.VideoCapture("video.mp4")

ret, frame = cap.read()

bbox = cv2.selectROI("Select Target", frame, False)
tracker = SpectreVideoTracker("spectre_got10k_mae_epoch_99.pth")

tracker.init(frame, bbox)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    box, conf = tracker.track(frame)

    x, y, w, h = box

    cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
    cv2.putText(frame, f"{conf:.2f}", (x, y-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    cv2.imshow("tracking", frame)

    if cv2.waitKey(1) == 27:
        break